<a href="https://colab.research.google.com/github/ikramkakar/PyTorchStudent-Pass-Fail-prediction/blob/main/PyTorch(Student_Pass_Fail_prediction).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Same dataset
hours = np.array([1,2,3,4,5,6,7,8,9,10]).reshape(-1,1)
result = np.array([0,0,0,0,1,1,1,1,1,1]).reshape(-1,1)

In [7]:
hours,result

(array([[ 1],
        [ 2],
        [ 3],
        [ 4],
        [ 5],
        [ 6],
        [ 7],
        [ 8],
        [ 9],
        [10]]),
 array([[0],
        [0],
        [0],
        [0],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1]]))

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    hours, result, test_size=0.2, random_state=42
)

In [9]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)

X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

In [11]:
model = nn.Sequential(
    nn.Linear(1, 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)

In [12]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [13]:
epochs = 50

for epoch in range(epochs):
    # Forward pass
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [10/50], Loss: 0.7207
Epoch [20/50], Loss: 0.6633
Epoch [30/50], Loss: 0.5750
Epoch [40/50], Loss: 0.4465
Epoch [50/50], Loss: 0.2918


In [14]:
with torch.no_grad():
    predictions = model(X_test)
    predicted = (predictions > 0.5).float()

    accuracy = (predicted == y_test).sum().item() / y_test.size(0)
    print("Accuracy:", accuracy)

Accuracy: 1.0


In [16]:
new_data = scaler.transform([[3]])
new_data = torch.FloatTensor(new_data)

with torch.no_grad():
    prediction = model(new_data)

    if prediction.item() > 0.5:
        print("Pass")
    else:
        print("Fail")

Fail


In [20]:

print(model)

Sequential(
  (0): Linear(in_features=1, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
  (5): Sigmoid()
)


In [23]:
!pip install torchsummary

In [25]:
from torchsummary import summary

# Print the model summary with an input size of (1, 1) since the model takes a single feature
summary(model, input_size=(1, 1))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                 [-1, 1, 8]              16
              ReLU-2                 [-1, 1, 8]               0
            Linear-3                 [-1, 1, 4]              36
              ReLU-4                 [-1, 1, 4]               0
            Linear-5                 [-1, 1, 1]               5
           Sigmoid-6                 [-1, 1, 1]               0
Total params: 57
Trainable params: 57
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------
